# BERT-QPP on TREC DL 2019 / 2020 (Google Colab)

Runs the BERT-QPP<sub>cross</sub> pipeline end-to-end against TREC DL 2019 and TREC DL 2019+2020, with all generated/downloaded artifacts persisted on Google Drive so re-running a session doesn't repeat expensive downloads.

**Pipeline:** mount Drive -> clone repo -> install deps -> fetch collection + pretrained model + qrels -> build test pickles -> predict QPP scores -> compute ground-truth per-query effectiveness -> correlate.

**Known data gap:** `run/result_files/monot5.1.tsv` and `run/result_files/e5_dl_1920.1.tsv` (and their 2019 counterparts) are currently **empty** in the repo (0 bytes) — the original extraction never populated them. Those two systems are skipped automatically below until you supply real run files. BM25 and ColBERT (and the PRF/RM3/SPLADE variants) have data and will run.

**Ground-truth AP/RR@10/nDCG@20 note:** the repo only ships *top-1-doc* run files (`qid<TAB>docid<TAB>rank`, no scores) — enough to build the QPP prediction inputs (Step 2/3), but not enough to score a system's actual retrieval effectiveness (Step 4 needs a full scored run). This notebook auto-generates a real, scored **BM25** run via PyTerrier's prebuilt MS MARCO index as a working end-to-end example. For ColBERT/SPLADE/monoT5/PRF/e5, bring your own scored `.res` file (see the last section) if you want their ground-truth correlation too.

## 1. Mount Drive & set paths
Edit `DRIVE_ROOT` if you want a different location.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/BERTQPP'          # persistent project root
REPO_DIR = f'{DRIVE_ROOT}/repo'                          # git checkout
DATA_DIR = f'{DRIVE_ROOT}/data'                          # collection.tsv, qrels
MODEL_DIR = f'{DRIVE_ROOT}/models'                        # trained/pretrained checkpoints
INDEX_DIR = f'{DRIVE_ROOT}/terrier_index'                 # cached PyTerrier index (for ground truth BM25)

for d in [DRIVE_ROOT, DATA_DIR, MODEL_DIR, INDEX_DIR]:
    os.makedirs(d, exist_ok=True)

## 2. Clone (or update) the repo on Drive

In [ ]:
if not os.path.isdir(f'{REPO_DIR}/.git'):
    !git clone https://github.com/Riddhi2587/BERTQPP.git "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd $REPO_DIR

## 3. Install dependencies

In [ ]:
!pip install -q sentence-transformers scipy pandas python-terrier gdown
!apt-get -qq install -y openjdk-11-jdk-headless > /dev/null
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

import pyterrier as pt
if not pt.java.started():
    pt.java.init()

## 4. Fetch the MS MARCO collection
Public direct download (no login), per the repo's README. ~2.9GB uncompressed — only downloads once thanks to the Drive cache.

In [ ]:
COLLECTION_PATH = f'{DATA_DIR}/collection.tsv'

if not os.path.exists(COLLECTION_PATH):
    !wget -q --show-progress -O /content/collectionandqueries.tar.gz \
        https://msmarco.blob.core.windows.net/msmarcoranking/collectionandqueries.tar.gz
    !tar -xzf /content/collectionandqueries.tar.gz -C /content collection.tsv
    !mv /content/collection.tsv "$COLLECTION_PATH"
    !rm /content/collectionandqueries.tar.gz
else:
    print(f'[INFO] Using cached collection at {COLLECTION_PATH}')

## 5. Fetch a trained BERT-QPP<sub>cross</sub> model
Uses the pretrained checkpoint linked from the repo's README (public Google Drive folder). If you'd rather train your own, see the **Train your own model** section at the bottom instead of running this cell.

In [ ]:
PRETRAINED_MODEL_DIR = f'{MODEL_DIR}/bertqpp_cross_pretrained'

if not os.path.isdir(PRETRAINED_MODEL_DIR) or not os.listdir(PRETRAINED_MODEL_DIR):
    os.makedirs(PRETRAINED_MODEL_DIR, exist_ok=True)
    !gdown --folder 'https://drive.google.com/drive/folders/1NDZzEpaay0cDumTKDUSMmv99sg9FyHrL' -O "$PRETRAINED_MODEL_DIR"
else:
    print(f'[INFO] Using cached model at {PRETRAINED_MODEL_DIR}')

MODEL_PATH = PRETRAINED_MODEL_DIR  # swap for your own checkpoint under MODEL_DIR if preferred

## 6. Build test pickles (Step 2)
One `.pkl` per run file, combining query text + top-retrieved-doc text. Skips the two known-empty run files.

In [ ]:
import glob, pathlib

def build_pkls(queries_tsv, run_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for run_file in sorted(glob.glob(f'{run_dir}/*.tsv')):
        if os.path.getsize(run_file) == 0:
            print(f'[SKIP] {run_file} is empty')
            continue
        base = pathlib.Path(run_file).stem
        out_file = f'{out_dir}/{base}.pkl'
        !python3 create_test_pkl_files.py \
            --collection "$COLLECTION_PATH" \
            --queries "{queries_tsv}" \
            --run "{run_file}" \
            --output "{out_file}"

PKL_DIR_19 = f'{DATA_DIR}/pklfiles/trecdl_19'
PKL_DIR_1920 = f'{DATA_DIR}/pklfiles/trecdl_1920'

build_pkls('trecdl2019_queries.tsv', 'run/result_files/2019', PKL_DIR_19)
build_pkls('trecdl1920_queries.tsv', 'run/result_files', PKL_DIR_1920)

## 7. Predict QPP scores (Step 3)

In [ ]:
def predict_all(pkl_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for pkl_file in sorted(glob.glob(f'{pkl_dir}/*.pkl')):
        base = pathlib.Path(pkl_file).stem
        out_file = f'{out_dir}/{base}_pred.txt'
        !python3 test_CE.py \
            --pkl "{pkl_file}" \
            --model "$MODEL_PATH" \
            --output "{out_file}"

PRED_DIR_19 = f'{DRIVE_ROOT}/results/trecdl_19'
PRED_DIR_1920 = f'{DRIVE_ROOT}/results/trecdl_1920'

predict_all(PKL_DIR_19, PRED_DIR_19)
predict_all(PKL_DIR_1920, PRED_DIR_1920)

## 8. Ground-truth example: BM25 on DL19 / DL19+20 (Step 4)
Uses PyTerrier's **prebuilt** MS MARCO passage index (downloaded, not rebuilt locally) plus `irds:` TREC DL topics/qrels — no manual qrels download needed.

In [ ]:
dataset = pt.get_dataset('msmarco_passage')
index = pt.IndexFactory.of(dataset.get_index('terrier_stemmed'))
bm25 = pt.terrier.Retriever(index, wmodel='BM25')

def bm25_run_to_res(dl_dataset_name, res_path):
    ds = pt.get_dataset(dl_dataset_name)
    topics = ds.get_topics()
    res = bm25.transform(topics)
    with open(res_path, 'w') as f:
        for _, row in res.iterrows():
            f.write(f"{row.qid} Q0 {row.docno} {row.rank} {row.score} bm25\n")
    return ds

RUNS_DIR = f'{DATA_DIR}/scored_runs'
os.makedirs(RUNS_DIR, exist_ok=True)

ds19 = bm25_run_to_res('irds:msmarco-passage/trec-dl-2019/judged', f'{RUNS_DIR}/BM25.19.res')
ds1920 = bm25_run_to_res('irds:msmarco-passage/trec-dl-2020/judged', f'{RUNS_DIR}/BM25.20.res')

qrels19_path = f'{DATA_DIR}/qrels.dl19.txt'
qrels20_path = f'{DATA_DIR}/qrels.dl20.txt'
pt.io.write_results(ds19.get_qrels(), qrels19_path, format='qrels')
pt.io.write_results(ds1920.get_qrels(), qrels20_path, format='qrels')

In [ ]:
AP_DIR = f'{DRIVE_ROOT}/ap_scores'
os.makedirs(AP_DIR, exist_ok=True)

!python3 compute_scores.py --res "{RUNS_DIR}/BM25.19.res" --qrels "{qrels19_path}" --out "{AP_DIR}/ap_BM25_dl19.json"
!python3 compute_scores.py --res "{RUNS_DIR}/BM25.20.res" --qrels "{qrels20_path}" --out "{AP_DIR}/ap_BM25_dl20.json"

## 9. Correlate predicted vs. actual (Step 5)

In [ ]:
print('--- DL19, BM25 ---')
!python3 evaluation.py --actual "{AP_DIR}/ap_BM25_dl19.json" --predicted "{PRED_DIR_19}/BM25.19.1_pred.txt" --target_metric AP

print('--- DL20, BM25 ---')
!python3 evaluation.py --actual "{AP_DIR}/ap_BM25_dl20.json" --predicted "{PRED_DIR_1920}/BM25.1920.1_pred.txt" --target_metric AP

## Bring your own run file for other systems
For ColBERT / monoT5 / SPLADE / RM3 / PRF / e5, you need a real scored `.res` file (`qid Q0 docno rank score runid`) from wherever you ran that system. Given `MY_RUN.res`:
```python
!python3 compute_scores.py --res MY_RUN.res --qrels "{qrels19_path}" --out "{AP_DIR}/ap_MYSYSTEM_dl19.json"
!python3 evaluation.py --actual "{AP_DIR}/ap_MYSYSTEM_dl19.json" --predicted "{PRED_DIR_19}/MYSYSTEM.1_pred.txt" --target_metric AP
```
The `_pred.txt` file already exists from Step 7 as long as `run/result_files/.../MYSYSTEM.1.tsv` had the top-1-doc data when Step 6 ran.

## (Optional) Train your own model instead of Step 5
`create_train_pkl_file.py` and `train_CE.py` currently have **hardcoded machine-specific paths** (`/media/pbclab/...`) left over from the original author's setup — edit those two files to point at your own `collection.tsv` / `pklfiles/` locations on Drive before running them here.